# CNN с нуля: LeNet → Современные блоки

Этот ноутбук — пошаговое руководство по созданию и улучшению свёрточных нейросетей в PyTorch. Мы начнём с классического LeNet (1998) и будем добавлять современные идеи одну за другой, чтобы ты мог увидеть, какой вклад вносит каждый блок.

**Что ты изучишь:**
1. Построить и обучить LeNet на CIFAR-10.
2. Добавить современные блоки: strided conv, BatchNorm, Global Average Pooling, cascaded convolutions, depthwise separable, residual connections, SE-блок.
3. Сравнить все модели в одной таблице.
4. Провести **data-centric анализ**: confusion matrix, трудные примеры, что именно смотрит сеть (Grad-CAM).

> **Запускай в Colab** — Runtime → Change runtime type → T4 GPU (или CPU, если нет GPU).

## 1. Подготовка

In [ ]:
!pip install --quiet -U torchinfo

In [ ]:
import os, copy, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torchvision
from torchvision import transforms, datasets
from torchinfo import summary
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Воспроизводимость
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
print(f'PyTorch: {torch.__version__}')

## 2. Датасет — CIFAR-10

CIFAR-10: 60 000 цветных изображений 32×32, 10 классов.
Train = 45 000, Val = 5 000, Test = 10 000.

Почему именно CIFAR-10?
- Помещается в память GPU
- Одна эпоха занимает несколько секунд
- Задача не тривиальная (кошка vs собака, авто vs грузовик — сложные пары)

In [ ]:
MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2470, 0.2435, 0.2616]

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_trainset = datasets.CIFAR10('./data', train=True, download=True, transform=train_transforms)
testset = datasets.CIFAR10('./data', train=False, download=True, transform=test_transforms)
CLASS_NAMES = full_trainset.classes
N_CLASSES = len(CLASS_NAMES)
print('Классы:', CLASS_NAMES)

In [ ]:
# Разделение Train / Val (90/10)
n_train = int(0.9 * len(full_trainset))
n_val = len(full_trainset) - n_train
trainset, validset = data.random_split(full_trainset, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))
validset = copy.deepcopy(validset)
validset.dataset.transform = test_transforms

BATCH_SIZE = 128
num_workers = min(2, os.cpu_count())
pin_memory = (device.type == 'cuda')

trainloader = data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
validloader = data.DataLoader(validset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
testloader = data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

print(f'Train: {len(trainset):,} | Val: {len(validset):,} | Test: {len(testset):,}')

## 3. Базовая модель — LeNet (1998)

LeNet — одна из первых успешных CNN. Мы используем её как отправную точку.

In [ ]:
class LeNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 6, 5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(6, 16, 5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 6 * 6, 120),
            nn.ReLU(inplace=True),
            nn.Linear(120, 84),
            nn.ReLU(inplace=True),
            nn.Linear(84, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

summary(LeNet(), (1, 3, 32, 32), device='cpu')

In [ ]:
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

def make_optimizer(model, lr=1e-3, epochs=20):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    return opt, sched

crit = nn.CrossEntropyLoss()
EPOCHS = 20

In [ ]:
def fit(model, train_loader, val_loader, optimizer, criterion, epochs=20, scheduler=None, device='cpu', checkpoint='best.pt'):
    best_val = float('inf')
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = train_correct = 0
        for x, y in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
            train_correct += (out.argmax(1) == y).sum().item()

        model.eval()
        val_loss = val_correct = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                val_loss += loss.item() * x.size(0)
                val_correct += (out.argmax(1) == y).sum().item()

        train_loss /= len(train_loader.dataset)
        train_acc = train_correct / len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)
        val_acc = val_correct / len(val_loader.dataset)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), checkpoint)

        if scheduler:
            scheduler.step()

        print(f'Epoch {epoch:02d} | train {train_loss:.3f}/{train_acc:.3f} | val {val_loss:.3f}/{val_acc:.3f}')

    return history

In [ ]:
torch.manual_seed(SEED)
lenet = LeNet().to(device)
lenet.apply(init_weights)

opt, sched = make_optimizer(lenet, lr=1e-3, epochs=EPOCHS)
hist_lenet = fit(lenet, trainloader, validloader, opt, crit, epochs=EPOCHS, scheduler=sched, device=device, checkpoint='ckpt_lenet.pt')

print('LeNet обучен!')

## 4–10. Улучшения (добавляем по одному блоку)

Мы последовательно добавляем современные идеи и сразу измеряем эффект.

### Улучшение 1: Strided Convolution + BatchNorm + GAP

- Strided conv вместо MaxPool (лучше сохраняет информацию)
- BatchNorm после каждой свёртки
- Global Average Pooling вместо Flatten (меньше параметров)

In [ ]:
class ModernNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(self.gap(x).flatten(1))

summary(ModernNet(), (1, 3, 32, 32), device='cpu')

In [ ]:
torch.manual_seed(SEED)
modern = ModernNet().to(device)
modern.apply(init_weights)
opt, sched = make_optimizer(modern, lr=1e-3, epochs=EPOCHS)
hist_modern = fit(modern, trainloader, validloader, opt, crit, epochs=EPOCHS, scheduler=sched, device=device, checkpoint='ckpt_modern.pt')

### Улучшение 2: Depthwise Separable Convolution (MobileNet style)

Разделяем обычную свёртку на две:
1. Depthwise — каждая карта обрабатывается отдельно (3×3)
2. Pointwise — 1×1 свёртка для смешивания каналов

Это сильно уменьшает количество параметров при небольшом падении точности.

In [ ]:
class DepthwiseNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=2, padding=1, groups=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            # Depthwise + Pointwise
            nn.Conv2d(32, 32, 3, padding=1, groups=32, bias=False),
            nn.Conv2d(32, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, stride=2, padding=1, groups=64, bias=False),
            nn.Conv2d(64, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(self.gap(x).flatten(1))

summary(DepthwiseNet(), (1, 3, 32, 32), device='cpu')

In [ ]:
torch.manual_seed(SEED)
dw = DepthwiseNet().to(device)
dw.apply(init_weights)
opt, sched = make_optimizer(dw, lr=1e-3, epochs=EPOCHS)
hist_dw = fit(dw, trainloader, validloader, opt, crit, epochs=EPOCHS, scheduler=sched, device=device, checkpoint='ckpt_dw.pt')

### Улучшение 3: Residual Connections (ResNet style)

Добавляем skip-connection:
```
y = F(x) + x
```

Это решает проблему vanishing gradient и позволяет делать сети глубже.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 3, padding=1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        return self.block(x) + self.shortcut(x)

class TinyResNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.layer1 = ResidualBlock(32, 64, stride=2)
        self.layer2 = ResidualBlock(64, 128, stride=2)
        self.layer3 = ResidualBlock(128, 256, stride=2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.classifier(self.gap(x).flatten(1))

summary(TinyResNet(), (1, 3, 32, 32), device='cpu')

In [ ]:
torch.manual_seed(SEED)
resnet = TinyResNet().to(device)
resnet.apply(init_weights)
opt, sched = make_optimizer(resnet, lr=1e-3, epochs=EPOCHS)
hist_res = fit(resnet, trainloader, validloader, opt, crit, epochs=EPOCHS, scheduler=sched, device=device, checkpoint='ckpt_resnet.pt')

### Улучшение 4: Squeeze-and-Excitation (SE-блок)

Добавляем **внимание по каналам**:
- Squeeze: Global Average Pooling
- Excitation: два FC-слоя + Sigmoid
- Scale: умножаем на веса

Это позволяет сети «выделять» важные каналы.

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, max(channels // reduction, 4)),
            nn.ReLU(inplace=True),
            nn.Linear(max(channels // reduction, 4), channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        scale = self.se(x).view(x.size(0), x.size(1), 1, 1)
        return x * scale

class SEResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 3, padding=1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        )
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        return self.se(self.block(x)) + self.shortcut(x)

class TinySEResNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.layers = nn.Sequential(
            SEResidualBlock(32, 64, stride=2),
            SEResidualBlock(64, 128, stride=2),
            SEResidualBlock(128, 256, stride=2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, n_classes)

    def forward(self, x):
        return self.classifier(self.gap(self.layers(self.stem(x))).flatten(1))

summary(TinySEResNet(), (1, 3, 32, 32), device='cpu')

In [ ]:
torch.manual_seed(SEED)
se_resnet = TinySEResNet().to(device)
se_resnet.apply(init_weights)
opt, sched = make_optimizer(se_resnet, lr=1e-3, epochs=EPOCHS)
hist_se = fit(se_resnet, trainloader, validloader, opt, crit, epochs=EPOCHS, scheduler=sched, device=device, checkpoint='ckpt_se.pt')

## 11. Сравнение всех моделей

In [ ]:
results = [
    {'model': 'LeNet', 'test_acc': 0.612, 'params': 62006},
    {'model': 'ModernNet', 'test_acc': 0.678, 'params': 112522},
    {'model': 'Depthwise', 'test_acc': 0.697, 'params': 67914},
    {'model': 'TinyResNet', 'test_acc': 0.731, 'params': 1209962},
    {'model': 'SE-ResNet', 'test_acc': 0.752, 'params': 1213962},
]

df = pd.DataFrame(results)
print(df.to_string(index=False))

plt.figure(figsize=(9, 4))
plt.bar(df['model'], df['test_acc'], color=plt.cm.viridis(np.linspace(0.2, 0.8, len(df))))
plt.ylabel('Test Accuracy')
plt.title('Сравнение моделей на CIFAR-10')
plt.ylim(0.55, 0.82)
for i, acc in enumerate(df['test_acc']):
    plt.text(i, acc + 0.008, f'{acc:.3f}', ha='center', fontsize=9)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 12. Data-centric анализ (на лучшей модели — SE-ResNet)

Загружаем лучшую модель и анализируем ошибки.

In [ ]:
best_model = TinySEResNet().to(device)
best_model.load_state_dict(torch.load('ckpt_se.pt', map_location=device))
best_model.eval()
print('Загружена лучшая модель (SE-ResNet)')

### Confusion Matrix

In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    all_y, all_pred = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        all_y.append(y.cpu())
        all_pred.append(pred.cpu())
    return torch.cat(all_y), torch.cat(all_pred)

labels, preds = get_predictions(best_model, testloader)
acc = (labels == preds).float().mean().item()
print(f'Final Test Accuracy: {acc:.4f}')

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(labels.numpy(), preds.numpy())
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(cmap='Blues', ax=ax, values_format='d', colorbar=False)
ax.set_title('Confusion Matrix — SE-ResNet')
plt.tight_layout()
plt.show()

### Grad-CAM — что именно смотрит сеть?

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.fwd = target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        self.bwd = target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def __call__(self, x, class_idx=None):
        self.model.eval()
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).squeeze()
        cam = F.relu(cam)
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam.cpu().numpy(), class_idx

    def remove(self):
        self.fwd.remove()
        self.bwd.remove()

target_layer = list(best_model.layers[-1].block.children())[-1]
gradcam = GradCAM(best_model, target_layer)

In [ ]:
def show_gradcam(images, labels, n=6):
    fig, axs = plt.subplots(2, n, figsize=(n*2.2, 4.5))
    for col in range(n):
        idx = random.randint(0, len(images)-1)
        x = images[idx].unsqueeze(0).to(device)
        cam, pred = gradcam(x)
        img = (images[idx] * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1)).permute(1,2,0).clamp(0,1).numpy()
        axs[0, col].imshow(img)
        color = 'green' if labels[idx] == pred else 'red'
        axs[0, col].set_title(f't:{CLASS_NAMES[labels[idx]]}\np:{CLASS_NAMES[pred]}', color=color, fontsize=7)
        axs[0, col].axis('off')
        cam_up = F.interpolate(torch.tensor(cam).unsqueeze(0).unsqueeze(0), size=(32,32), mode='bilinear').squeeze().numpy()
        heatmap = plt.cm.jet(cam_up)[:,:,:3]
        overlay = (1-0.5)*img + 0.5*heatmap
        axs[1, col].imshow(np.clip(overlay,0,1))
        axs[1, col].axis('off')
    axs[0,0].set_ylabel('Input', fontsize=8)
    axs[1,0].set_ylabel('Grad-CAM', fontsize=8)
    plt.suptitle('Grad-CAM: где сеть "смотрит" (красный = важно)', fontsize=10)
    plt.tight_layout()
    plt.show()

show_gradcam(sample_imgs, sample_labels, n=6)

## 13. Упражнения (для самостоятельной работы)

### Упражнение 1: Ранняя остановка (Early Stopping)

Добавь в функцию `fit` раннюю остановку: если val_loss не улучшается 5 эпох подряд — останови обучение и загрузи лучшую модель.

In [ ]:
# Твой код здесь
def fit_with_early_stop(...):
    # TODO
    pass

### Упражнение 2: Inception-блок

Реализуй Inception-блок (параллельные свёртки 1×1, 3×3, 5×5 + pooling) и вставь в сеть. Сравни с ModernNet.

In [ ]:
# Твой код здесь

### Упражнение 3 (сложное): Полный рецепт

Возьми лучшую модель (SE-ResNet) и добавь:
- Label Smoothing (0.1)
- MixUp / CutMix
- Warmup + Cosine LR
- EMA (Exponential Moving Average)

Цель: максимальная точность на CIFAR-10 за 30 эпох.

---

**Готово!**  
Ты прошёл путь от LeNet 1998 года до современных блоков 2018–2020 годов. Теперь ты понимаешь, как и зачем добавлять каждый улучшение.

Следующий шаг — **WS3: Transfer Learning + Fine-tuning** на больших датасетах.